# TN0 — Tái lập MobiVital và kiểm chứng pipeline thực nghiệm

Chạy một mạch trên Colab. Không cần notebook nào chạy trước.

## Ba kiểm tra

| | câu hỏi | bắt buộc đạt |
|---|---|---|
| **TN0a** | Tệp lựa chọn kênh tác giả cung cấp có tái hiện điểm công bố **0.819** không? | có |
| **TN0b** | Cùng tệp trọng số `.pth`, pipeline đồ án có chọn đúng 537/537 kênh giống pipeline MobiVital không? | có |
| **TN0c** | Train lại LSTM từ đầu thì đạt mức nào? | không — chỉ tham khảo |

## Luồng thực nghiệm

```
dữ liệu → train → tệp trọng số → chọn kênh → tính điểm → kết luận
```

Chạy hai lần trên cùng bộ dữ liệu:

| | code | các tệp chính |
|---|---|---|
| **pipeline MobiVital** | tác giả cung cấp | `training/autoreg_training.py`, `inference/mobivital_gen.py`, `inference/evaluate.py` |
| **pipeline đồ án** | trong `src/` | `training.py`, `scoring.py`, `results.py` |

Pipeline đồ án tồn tại vì code MobiVital chỉ chạy LSTM — `inference/mobivital_gen.py` dòng 152 ghi cứng `LSTMMultiStep(...)`, không hỗ trợ TCN. TN0 chứng minh nó cho ra đúng kết quả code gốc, rồi mới đổi LSTM sang TCN ở các thực nghiệm sau.

## Cách chia việc

```
notebook   →  trình bày luồng và gọi lệnh
scripts/   →  điều khiển thứ tự bước, bắt lỗi, giữ repo tác giả sạch
src/       →  model, train, chọn kênh, tính điểm
```

Mọi kết quả của TN0 nằm chung một thư mục `runs/tn0/` — checkpoint, đường cong loss, bảng lựa chọn kênh, điểm từng buổi ghi, metric. Cuối notebook nén thành `runs/tn0.zip` để tải về.


## 1. Chuẩn bị Colab


Mount Drive để lấy lại dữ liệu đã xử lý ở `DATA_PREPARE.ipynb`, và để cất tệp kết quả ở mục 6.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Tải mã nguồn đồ án về `/content/UWB_RADAR` rồi vào thư mục đó. Sau đó `setup_colab.py` clone MobiVital và **ghim commit `4319731d`** dùng cho mọi số liệu, rồi cài `einops`. Checkpoint giữ lại bằng cách nén ở mục 6 rồi tải về.


In [ ]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
# %cd phải ở notebook: os.chdir() trong script không đổi thư mục cho ô sau.
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


**Sửa duy nhất trong code tác giả.** `inference/mobivital_gen.py` nạp trọng số xong không gọi `model.eval()`, nên LSTM ở chế độ train và cuDNN cấp thêm 15.1 GB vùng dự trữ backward cho lô 6708 chuỗi của một buổi ghi. Đo thật: T4 15 GB và L4 22 GB đều tràn ngay ở buổi ghi 1/1874.

Bài báo ghi tác giả dùng GTX 1080 Ti chỉ 11 GB mà vẫn chạy được, vì `requirements.txt` của họ ghim `torch==2.3.0` — bản đó bỏ qua vùng dự trữ khi đang trong `torch.no_grad()`. PyTorch hiện hành cấp theo cờ `model.training`. Không cài lại được vì `torch==2.3.0` không có wheel cho Python 3.13.

`patch_eval.py` thêm **đúng một dòng** `model.eval()`, kèm chú thích đánh dấu trong file. Model chỉ gồm `nn.LSTM(dropout=0)` và `nn.Linear` nên train và eval cho forward giống hệt — **không đổi kết quả**, chỉ đổi cách xin bộ nhớ.


In [ ]:
!python scripts/mobivital/patch_eval.py


## 2. Dữ liệu chung

Một bản CSV duy nhất, đặt trong thư mục MobiVital. Hai pipeline đọc chung bản đó:

```
external/mobivital/dataset/mobivital/tripod/*.csv     1874 tệp
        |
        +-- prep_breath_final.py cua tac gia  ->  external/mobivital/data_final/*.npy
        |
        +-- scripts/make_npz.py cua do an     ->  data/processed/by_user/*.npz
                        |
                  phai khop TUNG BYTE
```

Mỗi lệnh dưới tự bỏ qua nếu đã đủ tệp, nên chạy lại notebook không mất thời gian làm lại. Nhưng một bước đang dở thì phải làm lại từ đầu bước đó.


Tải 5.7 GB từ Zenodo rồi giải nén thành 1874 tệp CSV.


In [ ]:
!python scripts/download_dataset.py


Vá 52 tên tệp lỗi thời để `evaluate.py` mở được, và giấu dữ liệu khỏi git của tác giả.


In [ ]:
!python scripts/mobivital/setup_dataset.py


Chạy `prep_breath_final.py` của tác giả: CSV → `data_final/*.npy` (ABCDEFKL 1289 buổi ghi, GHIJ 537 buổi ghi).


In [ ]:
!python scripts/mobivital/run_tn0.py --case prep


Lấy `by_user/` và `windows/` từ Drive nếu `DATA_PREPARE.ipynb` đã cất lên — đỡ khoảng 15 phút. Không có thì hai bước sau tự dựng lại.


In [ ]:
!python scripts/restore_processed_data_on_drive.py


Nếu bước trên chưa lấy được thì dựng lại: đọc đúng bộ CSV đó bằng code đồ án, gom theo từng người → `by_user/*.npz`. Đã đủ 12 tệp thì tự bỏ qua.


In [ ]:
!python scripts/make_npz.py


So hai bên. Phải khớp **từng byte**: ABCDEFKL 1289/1289, GHIJ 537/537. Không khớp thì dừng.


In [ ]:
!python scripts/check_data.py


Cắt sẵn cửa sổ train cho pipeline đồ án: 200 mẫu vào → 25 mẫu phải đoán. Đã cắt đủ thì tự bỏ qua.


In [ ]:
!python scripts/make_windows.py


## 3. Pipeline MobiVital

Chạy đúng lệnh trong README của tác giả. Script `scripts/mobivital/run_tn0.py` in nguyên văn từng lệnh trước khi chạy, nên đọc output là thấy đủ chuỗi lệnh gốc.

Hai việc script làm thêm:

1. `mobivital_gen.py` ghi tệp lựa chọn kênh đè lên một tệp **có sẵn trong repo tác giả**, nên sau mỗi lần chạy phải chép kết quả ra `runs/tn0/` rồi khôi phục tệp gốc.

2. Kiểm repo tác giả chỉ bị đổi đúng một tệp đã vá ở mục 1, và chỉ trong khối đánh dấu. Đụng tệp nào khác là dừng.


**TN0a** — tính điểm từ tệp lựa chọn kênh tác giả cung cấp. Chưa đụng model: kênh khoảng cách và phép biến đổi đã ghi sẵn trong tệp.


In [ ]:
!python scripts/mobivital/run_tn0.py --case a


**TN0b** — dùng tệp trọng số tác giả phát hành để chọn kênh trên G H I J. Mỗi buổi ghi: dựng 240 ứng viên (120 kênh khoảng cách × 2 phép biến đổi) → loại các ứng viên bị phát hiện đảo chiều bằng hàm `invert_detector()` của MobiVital → LSTM dự báo → chọn ứng viên có tổng Pearson cao nhất. Bước chọn **không nhìn nhịp thở thật**.


In [ ]:
!python scripts/mobivital/run_tn0.py --case b


**TN0c** — train lại LSTM từ đầu bằng chính vòng train của tác giả, cấu hình `checkpoints/optimal_params.json`: 20 epoch, Adam lr 1e-4, batch 64, MSE.


In [ ]:
!python scripts/mobivital/run_tn0.py --case c


## 4. Pipeline đồ án

Ba kiểm tra như trên, bằng code trong `src/`:

| kiểm tra | pipeline MobiVital | pipeline đồ án |
|---|---|---|
| tính điểm từ tệp lựa chọn kênh | `inference/evaluate.py` | `scoring.score_from_txt` |
| chọn kênh từ tệp trọng số | `inference/mobivital_gen.py` | `scoring.score_all` |
| train lại LSTM | `training/autoreg_training.py` | `training.train` |

Mỗi `--case` thêm đúng một bộ phận, nên bộ phận nào sai thì lộ ra ở đúng kiểm tra đó.


**TN0a** — chỉ dùng hàm tính điểm, chưa chạy model.


In [ ]:
!python scripts/run_tn0.py --case a


**TN0b** — thêm bộ chọn kênh, vẫn dùng tệp trọng số tác giả phát hành.


In [ ]:
!python scripts/run_tn0.py --case b


**TN0c** — thêm vòng train, cùng cấu hình MobiVital công bố.


In [ ]:
!python scripts/run_tn0.py --case c


## 5. So sánh hai pipeline

Lệnh dưới kiểm bốn điều và trả mã lỗi khác 0 nếu có điều bắt buộc không đạt:

1. **TN0a tái hiện bài báo** — điểm pipeline MobiVital lệch `0.819` dưới `0.001`.
2. **TN0a hai pipeline khớp** — chênh lệch trên **từng** buổi ghi dưới `1e-9`, không chỉ so điểm trung bình.
3. **TN0b hai pipeline khớp** — 537/537 buổi ghi chọn cùng kênh, chênh lệch từng buổi dưới `1e-9`.
4. **Repo MobiVital sạch** — `git status --porcelain` trống.

TN0c chỉ tham khảo vì đây là hai lần train độc lập; không yêu cầu trọng số và điểm số giống tuyệt đối.


In [ ]:
!python scripts/run_tn0.py --compare


## 6. Lưu kết quả

Nén cả thư mục `runs/tn0/` thành `runs/tn0.zip`, và chép sang Drive nếu đã mount. Giải nén lại bằng `unzip tn0.zip -d runs/`.


In [ ]:
!python scripts/save_results.py tn0
